# Mini-Projekt 1: Regression von Hand

Aufgabenbeschreibung und Erfolgskriterien findest du auf der Projektseite der Kurs-Website.
Ziel: eine lineare Regression mit Gradientenabstieg **nur mit NumPy** – und am Ende derselbe Test-Fehler wie scikit-learn.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (7, 4)
rng = np.random.default_rng(42)

In [ ]:
from sklearn.datasets import load_diabetes

data = load_diabetes(scaled=False)
X, y = data.data, data.target
namen = data.feature_names
print(X.shape, y.shape)
print(namen)

## Schritt 1: Erkunden

In [ ]:
# TODO: Mittelwert und Standardabweichung jedes Features ausgeben
...
# TODO: Korrelation jedes Features mit y; das stärkste Feature bestimmen
korrelationen = ...
j_best = np.argmax(np.abs(korrelationen))
print("stärkste Korrelation:", namen[j_best], round(korrelationen[j_best], 3))
plt.scatter(X[:, j_best], y, s=10); plt.xlabel(namen[j_best]); plt.ylabel("Krankheitsfortschritt"); plt.show()

## Schritt 2: Aufteilen in Training und Test (80/20)

In [ ]:
# TODO: zufällige Permutation mit Seed 42, die ersten 80 % sind Training
...
X_train, X_test, y_train, y_test = X[train_idx], X[test_idx], y[train_idx], y[test_idx]
print(X_train.shape, X_test.shape)

## Schritt 3: Standardisieren

Mittelwert und Standardabweichung **nur aus den Trainingsdaten** – sonst fließt Information aus dem Test-Satz ins Training (*Data Leakage*).

In [ ]:
def standardisierer(X_train):
    # TODO: Mittelwert und Std der Trainingsdaten berechnen und eine Transformationsfunktion zurückgeben
    ...

transform = standardisierer(X_train)
Z_train, Z_test = transform(X_train), transform(X_test)
print(np.round(Z_train.mean(axis=0), 6))
print(np.round(Z_train.std(axis=0), 6))

## Schritt 4 und 5: Gradient implementieren und prüfen

In [ ]:
def vorhersage(X, w, b):
    return X @ w + b

def mse(X, y, w, b):
    return np.mean((vorhersage(X, w, b) - y) ** 2)

def mse_gradient(X, y, w, b):
    # TODO: Gradienten nach w (Vektor) und b (Zahl)
    ...

def num_gradient(X, y, w, b, h=1e-5):
    gw = np.zeros_like(w)
    for i in range(len(w)):
        e = np.zeros_like(w); e[i] = h
        gw[i] = (mse(X, y, w + e, b) - mse(X, y, w - e, b)) / (2 * h)
    gb = (mse(X, y, w, b + h) - mse(X, y, w, b - h)) / (2 * h)
    return gw, gb

w_test, b_test = rng.normal(size=X.shape[1]), 10.0
gw, gb = mse_gradient(Z_train, y_train, w_test, b_test)
gw_n, gb_n = num_gradient(Z_train, y_train, w_test, b_test)
print("relative Abweichung:", np.max(np.abs(gw - gw_n)) / np.max(np.abs(gw)), abs(gb - gb_n) / abs(gb))

## Schritt 4 (Fortsetzung): Modell nur mit BMI

In [ ]:
def train(X, y, lr, epochen):
    w, b = np.zeros(X.shape[1]), 0.0
    verlauf = []
    for _ in range(epochen):
        # TODO: ein Gradientenschritt, dann Verlust protokollieren
        ...
    return w, b, verlauf

j_bmi = namen.index("bmi")
w1, b1, _ = train(Z_train[:, [j_bmi]], y_train, lr=0.1, epochen=500)
plt.scatter(Z_train[:, j_bmi], y_train, s=10)
t = np.linspace(-2.5, 3.5, 2); plt.plot(t, w1[0] * t + b1, color="tab:orange")
plt.xlabel("BMI (standardisiert)"); plt.ylabel("y"); plt.show()
print("Test-MSE nur mit BMI:", mse(Z_test[:, [j_bmi]], y_test, w1, b1))

## Schritt 6: Alle Features, verschiedene Lernraten

In [ ]:
with np.errstate(over="ignore", invalid="ignore"):   # η = 1.0 divergiert absichtlich
    for lr in [0.001, 0.01, 0.1, 1.0]:
        w, b, verlauf = train(Z_train, y_train, lr, epochen=500)
        verlauf = np.nan_to_num(verlauf, nan=1e9, posinf=1e9)   # divergierte Werte für den Plot kappen
        plt.plot(np.minimum(verlauf, 1e9), label=f"η = {lr}")
plt.yscale("log"); plt.ylim(2e3, 1e5); plt.xlabel("Epoche"); plt.ylabel("Trainings-MSE"); plt.legend(); plt.show()

$\eta = 1{,}0$ divergiert, obwohl bei einem einzelnen standardisierten Feature sogar Lernraten bis 1 funktionieren würden.
Die maximale stabile Lernrate hängt von der stärksten Krümmung der Verlustlandschaft ab: Einige Blutwerte (`s1`, `s2`) sind stark
miteinander korreliert, was die Landschaft in eine Richtung sehr steil macht. Hier liegt die Grenze bei etwa $\eta \approx 0{,}26$.

Wir trainieren das finale Modell mit $\eta = 0{,}1$ und genügend Epochen.

In [ ]:
w_gd, b_gd, verlauf = train(Z_train, y_train, lr=0.1, epochen=20_000)
mse_test_gd = mse(Z_test, y_test, w_gd, b_gd)
baseline = np.mean((y_test - y_train.mean()) ** 2)
print(f"Test-MSE: {mse_test_gd:.1f}   Baseline (immer Mittelwert): {baseline:.1f}   R² = {1 - mse_test_gd / np.var(y_test):.3f}")

## Schritt 7: Mini-Batch-Gradientenabstieg

In [ ]:
def train_minibatch(X, y, lr, epochen, batch=32, seed=0):
    r_gen = np.random.default_rng(seed)
    w, b = np.zeros(X.shape[1]), 0.0
    verlauf = []
    for _ in range(epochen):
        # TODO: Daten mischen, in Batches durchgehen, auf jedem Batch einen Schritt machen
        ...
        verlauf.append(mse(X, y, w, b))
    return w, b, verlauf

_, _, v_batch = train(Z_train, y_train, 0.1, 300)
w_mb, b_mb, v_mb = train_minibatch(Z_train, y_train, 0.01, 300)
plt.plot(v_batch, label="Batch-GD (η=0.1)"); plt.plot(v_mb, label="Mini-Batch 32 (η=0.01)")
plt.yscale("log"); plt.xlabel("Epoche"); plt.ylabel("Trainings-MSE"); plt.legend(); plt.show()

## Schritt 8: Vergleich mit scikit-learn

In [ ]:
from sklearn.linear_model import LinearRegression

sk = LinearRegression().fit(Z_train, y_train)
mse_test_sk = np.mean((sk.predict(Z_test) - y_test) ** 2)
print(f"{'Feature':6s} {'unser GD':>10s} {'sklearn':>10s}")
for name, a, c in zip(namen, w_gd, sk.coef_):
    print(f"{name:6s} {a:10.3f} {c:10.3f}")
print(f"Bias   {b_gd:10.3f} {sk.intercept_:10.3f}")
print(f"Test-MSE: unser {mse_test_gd:.2f}  sklearn {mse_test_sk:.2f}")
print("max. relative Abweichung der Gewichte:", np.max(np.abs(w_gd - sk.coef_) / np.abs(sk.coef_)))

## Schritt 9: Ohne Standardisierung

In [ ]:
# TODO: Trainiere auf den unskalierten X_train mit verschiedenen Lernraten (200 Epochen) und gib den letzten Verlust aus
...

Ohne Skalierung reichen die Features von ca. 1 (Geschlecht) bis über 200 (Cholesterin). Die Verlustlandschaft ist extrem langgezogen:
Für die großen Features ist sie so steil, dass nur winzige Lernraten stabil sind – und mit denen kommt man in den flachen Richtungen kaum voran.

## Schritt 10: Deuten

In [ ]:
# TODO: Die drei Features mit den betragsmäßig größten Gewichten (standardisiert) ausgeben
...

Bei standardisierten Features bedeutet ein Gewicht: "Wie stark ändert sich die Vorhersage, wenn dieses Feature um eine Standardabweichung steigt?"
Damit sind Gewichte vergleichbar. Bei unskalierten Features hängt das Gewicht von der Einheit ab (mg/dl oder g/l?) und ist nicht vergleichbar.

**Vorsicht:** Bei stark korrelierten Features (hier `s1` und `s2`) können sich große Gewichte mit entgegengesetztem Vorzeichen gegenseitig aufheben.
Dieses Problem heißt **Multikollinearität** – Regularisierung (Ridge, Modul 3) hilft dagegen.